# Gene Usage Batch-Correction Benchmark on AIRR COVID-19 (HF)

This notebook benchmarks batch correction on real data from `datasets/isalgo/airr_covid19`.

Pipeline:
1. Download/cache dataset in `notebooks/assets/large`.
2. Load metadata and repertoires with `RepertoireDataset.from_folder_polars(...)`.
3. Compute VJ usage correction (`compute_batch_corrected_gene_usage`).
4. Reproduce segment-usage style analyses: PCA, batch separability metrics, clustermaps, highlighted genes.
5. Re-run after `resample_to_gene_usage` and `filter_functional`.
6. Report final sample/batch/locus stats.

In [ ]:
# If needed, uncomment and run once:
# %pip install -q huggingface_hub datasets polars seaborn scikit-learn

from __future__ import annotations

from pathlib import Path
import os
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

from huggingface_hub import snapshot_download

from mir.common.parser import VDJtoolsParser
from mir.common.repertoire_dataset import RepertoireDataset
from mir.common.sampling import resample_to_gene_usage
from mir.common.filter import filter_functional
from mir.basic.gene_usage import compute_batch_corrected_gene_usage, GeneUsage

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
asset_root = repo_root / 'notebooks' / 'assets' / 'large'
asset_root.mkdir(parents=True, exist_ok=True)

hf_local_dir = asset_root / 'airr_covid19'

if not hf_local_dir.exists() or not any(hf_local_dir.iterdir()):
    print('Downloading dataset from Hugging Face...')
    snapshot_download(
        repo_id='isalgo/airr_covid19',
        repo_type='dataset',
        local_dir=str(hf_local_dir),
        local_dir_use_symlinks=False,
    )
else:
    print(f'Using cached dataset at: {hf_local_dir}')

hf_local_dir

In [ ]:
# Locate metadata file (robust to internal folder layout)
meta_candidates = sorted(hf_local_dir.rglob('metadata.tsv')) + sorted(hf_local_dir.rglob('*metadata*.tsv'))
if not meta_candidates:
    raise FileNotFoundError('Could not locate metadata TSV in the downloaded dataset directory.')

metadata_path = meta_candidates[0]
data_root = metadata_path.parent
print('metadata_path =', metadata_path)
print('data_root =', data_root)

meta_pl = pl.read_csv(metadata_path, separator='	', infer_schema_length=20_000)
print('metadata columns:', meta_pl.columns)

required_cols = {'sample_id', 'batch_id', 'file_name', 'locus'}
missing_cols = required_cols.difference(set(meta_pl.columns))
if missing_cols:
    raise ValueError(f'Metadata is missing required columns: {sorted(missing_cols)}')

meta_pl.head()

In [ ]:
# Build dataset with parser + polars metadata loader; skip missing files as requested.
parser = VDJtoolsParser()
dataset = RepertoireDataset.from_folder_polars(
    data_root,
    parser=parser,
    metadata_file=metadata_path.name,
    file_name_column='file_name',
    sample_id_column='sample_id',
    metadata_sep='	',
    skip_missing_files=True,
)

print(f'Loaded samples: {len(dataset.samples)}')

# Keep only samples with at least one non-empty TRB/TRA locus
valid_loci = {'TRA', 'TRB'}
filtered_samples = {}
for sid, srep in dataset.samples.items():
    loci = {loc: lr for loc, lr in srep.loci.items() if loc in valid_loci and len(lr.clonotypes) > 0}
    if loci:
        srep.loci = loci
        filtered_samples[sid] = srep

dataset.samples = filtered_samples
dataset.metadata = {sid: dataset.metadata[sid] for sid in dataset.samples.keys() if sid in dataset.metadata}
print(f'Samples after locus filter: {len(dataset.samples)}')

In [ ]:
# Sanity checks: d='.' should be normalized to empty by parser._gene_str
dot_d_count = 0
empty_d_count = 0
total_clones = 0

for s in dataset.samples.values():
    for lr in s.loci.values():
        for c in lr.clonotypes:
            total_clones += 1
            if c.d_gene == '.':
                dot_d_count += 1
            if c.d_gene == '':
                empty_d_count += 1

print('Total clonotypes:', total_clones)
print('Clonotypes with d_gene==.:', dot_d_count)
print('Clonotypes with empty d_gene:', empty_d_count)
assert dot_d_count == 0, 'Expected d_gene "." to be normalized to empty string.'

In [ ]:
def _gene_to_str(g):
    if isinstance(g, tuple):
        return '|'.join(map(str, g))
    return str(g)

def _build_sample_matrix(df: pd.DataFrame, value_col: str, locus: str = 'TRB') -> pd.DataFrame:
    d = df[df['locus'] == locus].copy()
    d['gene_str'] = d['gene'].map(_gene_to_str)
    mat = d.pivot_table(index='sample_id', columns='gene_str', values=value_col, aggfunc='mean', fill_value=0.0)
    # Normalize each sample to sum=1 to compare compositions
    row_sum = mat.sum(axis=1).replace(0, np.nan)
    mat = mat.div(row_sum, axis=0).fillna(0.0)
    return mat

def _pca_df(mat: pd.DataFrame, metadata_df: pd.DataFrame, n_components: int = 2) -> pd.DataFrame:
    X = mat.values
    Xs = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
    pca = PCA(n_components=n_components, random_state=42)
    Z = pca.fit_transform(Xs)
    out = pd.DataFrame(Z, index=mat.index, columns=[f'PC{i+1}' for i in range(n_components)])
    out = out.merge(metadata_df[['sample_id', 'batch_id']], left_index=True, right_on='sample_id', how='left')
    out['explained_var_pc1'] = pca.explained_variance_ratio_[0]
    out['explained_var_pc2'] = pca.explained_variance_ratio_[1] if n_components > 1 else np.nan
    return out

def _batch_separability_metrics(embedding_df: pd.DataFrame) -> dict:
    lab = embedding_df['batch_id'].astype(str).values
    pts = embedding_df[['PC1', 'PC2']].values
    uniq = pd.unique(lab)

    if len(uniq) < 2:
        return {'silhouette_batch': np.nan, 'kmeans_ari': np.nan, 'kmeans_nmi': np.nan}

    sil = silhouette_score(pts, lab)
    km = KMeans(n_clusters=len(uniq), random_state=42, n_init='auto').fit(pts)
    ari = adjusted_rand_score(lab, km.labels_)
    nmi = normalized_mutual_info_score(lab, km.labels_)
    return {'silhouette_batch': sil, 'kmeans_ari': ari, 'kmeans_nmi': nmi}

In [ ]:
# VJ correction on TRB
corr_vj = compute_batch_corrected_gene_usage(
    dataset, batch_field='batch_id', scope='vj', weighted=True, pseudocount=1.0, z_cap=6.0
)
corr_vj['zsig'] = 1.0 / (1.0 + np.exp(-corr_vj['z']))

meta_df = dataset.metadata_df.reset_index(drop=True)

mat_before_vj = _build_sample_matrix(corr_vj, value_col='p', locus='TRB')
mat_after_vj = _build_sample_matrix(corr_vj, value_col='pfinal', locus='TRB')

# pfinal may include sign changes due to the transformation; use zsig for stable PCA embedding.
mat_after_vj_stable = _build_sample_matrix(corr_vj, value_col='zsig', locus='TRB')

pca_before = _pca_df(mat_before_vj, meta_df)
pca_after = _pca_df(mat_after_vj_stable, meta_df)

met_before = _batch_separability_metrics(pca_before)
met_after = _batch_separability_metrics(pca_after)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
palette = sns.color_palette('tab20', n_colors=max(2, pca_before['batch_id'].nunique()))

sns.scatterplot(data=pca_before, x='PC1', y='PC2', hue='batch_id', ax=axes[0], palette=palette, s=35, alpha=0.85)
axes[0].set_title('PCA on VJ Usage (Before Correction)')
axes[0].legend(title='batch', bbox_to_anchor=(1.02, 1), loc='upper left')

sns.scatterplot(data=pca_after, x='PC1', y='PC2', hue='batch_id', ax=axes[1], palette=palette, s=35, alpha=0.85)
axes[1].set_title('PCA on VJ Usage (After Correction; zsig)')
axes[1].legend(title='batch', bbox_to_anchor=(1.02, 1), loc='upper left')

print('Batch separability metrics:')
print('Before:', met_before)
print('After :', met_after)
plt.show()

In [ ]:
# Segment-level (V only) analysis and highlighted genes
corr_v = compute_batch_corrected_gene_usage(
    dataset, batch_field='batch_id', scope='v', weighted=True, pseudocount=1.0, z_cap=6.0
)
corr_v['zsig'] = 1.0 / (1.0 + np.exp(-corr_v['z']))

highlight_genes = ['TRBV6-2/TRBV6-3', 'TRBV28', 'TRBV4-3', 'TRBV3-2']

# Normalize gene names for matching (strip allele and uppercase)
def _norm_v_name(x: str) -> str:
    s = str(x).upper().split('*')[0]
    return s

corr_v_plot = corr_v.copy()
corr_v_plot['gene_norm'] = corr_v_plot['gene'].map(_norm_v_name)

# Handle TRBV6-2/TRBV6-3 aggregate by combining both
corr_v_plot['gene_plot'] = corr_v_plot['gene_norm']
corr_v_plot.loc[corr_v_plot['gene_norm'].isin(['TRBV6-2', 'TRBV6-3']), 'gene_plot'] = 'TRBV6-2/TRBV6-3'

before_v = corr_v_plot.pivot_table(index='sample_id', columns='gene_plot', values='p', aggfunc='mean', fill_value=0.0)
after_v = corr_v_plot.pivot_table(index='sample_id', columns='gene_plot', values='zsig', aggfunc='mean', fill_value=0.0)

available_highlight = [g for g in highlight_genes if g in before_v.columns]
if not available_highlight:
    # fallback: show top variable V genes if specific set is not fully present
    var_order = before_v.var(axis=0).sort_values(ascending=False)
    available_highlight = list(var_order.head(20).index)

before_h = before_v[available_highlight]
after_h = after_v[available_highlight]

batch_series = meta_df.set_index('sample_id').loc[before_h.index, 'batch_id'].astype(str)
batch_palette = dict(zip(sorted(batch_series.unique()), sns.color_palette('Set2', n_colors=batch_series.nunique())))
row_colors = batch_series.map(batch_palette)

sns.clustermap(before_h, cmap='mako', row_colors=row_colors, figsize=(10, 8), standard_scale=1)
plt.suptitle('Clustermap Before Correction (Highlighted V Genes)', y=1.02)
plt.show()

sns.clustermap(after_h, cmap='crest', row_colors=row_colors, figsize=(10, 8), standard_scale=1)
plt.suptitle('Clustermap After Correction (Highlighted V Genes; zsig)', y=1.02)
plt.show()

In [ ]:
# Distribution plots for highlighted genes: persistence of bi/multimodal behavior
dist_df = corr_v_plot.merge(meta_df[['sample_id', 'batch_id']], on='sample_id', how='left')

plot_genes = [g for g in ['TRBV6-2/TRBV6-3', 'TRBV28', 'TRBV4-3', 'TRBV3-2'] if g in dist_df['gene_plot'].unique()]
if not plot_genes:
    plot_genes = sorted(dist_df['gene_plot'].unique())[:4]

fig, axes = plt.subplots(len(plot_genes), 2, figsize=(12, 3.0 * len(plot_genes)), constrained_layout=True)
if len(plot_genes) == 1:
    axes = np.array([axes])

for i, g in enumerate(plot_genes):
    d = dist_df[dist_df['gene_plot'] == g]
    sns.histplot(data=d, x='p', hue='batch_id', bins=40, stat='density', common_norm=False, ax=axes[i, 0], alpha=0.45)
    axes[i, 0].set_title(f'{g} before correction (p)')

    sns.histplot(data=d, x='zsig', hue='batch_id', bins=40, stat='density', common_norm=False, ax=axes[i, 1], alpha=0.45)
    axes[i, 1].set_title(f'{g} after correction (zsig)')

plt.show()

In [ ]:
# Resample each sample to VJ pooled target and then filter non-functional
corr_vj_for_target = compute_batch_corrected_gene_usage(
    dataset, batch_field='batch_id', scope='vj', weighted=True, pseudocount=1.0, z_cap=6.0
)

target_scale = 10_000
target_vj_usage = {
    row['gene']: max(1, int(round(float(row['pavg']) * target_scale)))
    for _, row in (
        corr_vj_for_target[corr_vj_for_target['locus'] == 'TRB']
        .drop_duplicates('gene')[['gene', 'pavg']]
        .iterrows()
    )
}

resampled_samples = {}
resampled_filtered_samples = {}

for sid, srep in dataset.samples.items():
    if 'TRB' not in srep.loci:
        continue

    trb = srep.loci['TRB']
    trb_resampled = resample_to_gene_usage(
        trb,
        target_vj_usage,
        scope='vj',
        weighted=True,
        random_seed=42,
    )

    s_res = type(srep)(loci={'TRB': trb_resampled}, sample_id=sid, sample_metadata=dict(srep.sample_metadata))
    s_res_f = filter_functional(s_res)

    resampled_samples[sid] = s_res
    resampled_filtered_samples[sid] = s_res_f

dataset_resampled = RepertoireDataset(samples=resampled_samples)
dataset_resampled_filtered = RepertoireDataset(samples=resampled_filtered_samples)

print('Resampled samples:', len(dataset_resampled.samples))
print('Resampled + non-functional-filtered samples:', len(dataset_resampled_filtered.samples))

# Compare highlighted-gene distributions after resampling and filtering
corr_resampled_v = compute_batch_corrected_gene_usage(
    dataset_resampled, batch_field='batch_id', scope='v', weighted=True
)
corr_resampled_v['zsig'] = 1.0 / (1.0 + np.exp(-corr_resampled_v['z']))

corr_resampled_filtered_v = compute_batch_corrected_gene_usage(
    dataset_resampled_filtered, batch_field='batch_id', scope='v', weighted=True
)
corr_resampled_filtered_v['zsig'] = 1.0 / (1.0 + np.exp(-corr_resampled_filtered_v['z']))


def prep_dist(d: pd.DataFrame, label: str) -> pd.DataFrame:
    x = d.copy()
    x['gene_norm'] = x['gene'].astype(str).str.upper().str.split('*').str[0]
    x['gene_plot'] = x['gene_norm']
    x.loc[x['gene_norm'].isin(['TRBV6-2', 'TRBV6-3']), 'gene_plot'] = 'TRBV6-2/TRBV6-3'
    x['dataset'] = label
    return x


dist_comp = pd.concat([
    prep_dist(corr_resampled_v, 'resampled'),
    prep_dist(corr_resampled_filtered_v, 'resampled+filter_functional'),
], ignore_index=True)

genes_to_show = [g for g in ['TRBV6-2/TRBV6-3', 'TRBV28', 'TRBV4-3', 'TRBV3-2'] if g in set(dist_comp['gene_plot'])]
if not genes_to_show:
    genes_to_show = sorted(dist_comp['gene_plot'].unique())[:4]

fig, axes = plt.subplots(len(genes_to_show), 1, figsize=(10, 3.2 * len(genes_to_show)), constrained_layout=True)
if len(genes_to_show) == 1:
    axes = [axes]

for ax, g in zip(axes, genes_to_show):
    d = dist_comp[dist_comp['gene_plot'] == g]
    sns.histplot(data=d, x='p', hue='dataset', bins=40, stat='density', common_norm=False, alpha=0.4, ax=ax)
    ax.set_title(f'{g}: resample_to_gene_usage and filter_functional')

plt.show()

# Final cohort statistics: samples, batches, loci
sample_rows = []
for sid, srep in dataset.samples.items():
    loci_present = sorted(list(srep.loci.keys()))
    sample_rows.append({
        'sample_id': sid,
        'batch_id': str(dataset.metadata[sid].get('batch_id', 'NA')),
        'n_loci_loaded': len(loci_present),
        'loci': ','.join(loci_present),
        'n_clonotypes_total': int(sum(len(lr.clonotypes) for lr in srep.loci.values())),
        'n_duplicates_total': int(sum(sum(c.duplicate_count for c in lr.clonotypes) for lr in srep.loci.values())),
    })

sample_stats = pd.DataFrame(sample_rows)
batch_stats = sample_stats.groupby('batch_id').agg(
    n_samples=('sample_id', 'nunique'),
    n_clonotypes=('n_clonotypes_total', 'sum'),
    n_duplicates=('n_duplicates_total', 'sum'),
).sort_values('n_samples', ascending=False)

locus_counter = Counter()
for s in dataset.samples.values():
    for loc in s.loci.keys():
        locus_counter[loc] += 1
locus_stats = pd.DataFrame({
    'locus': list(locus_counter.keys()),
    'n_samples_with_locus': list(locus_counter.values())
}).sort_values('n_samples_with_locus', ascending=False)

print('=== Final Statistics ===')
print('Loaded samples:', sample_stats['sample_id'].nunique())
print('Loaded batches:', sample_stats['batch_id'].nunique())
print('Loci observed:', sorted(locus_counter.keys()))
print('Samples with >2 loci (should be 0 or rare):', int((sample_stats['n_loci_loaded'] > 2).sum()))

sample_stats.head(), batch_stats.head(20), locus_stats